Need imports first 

In [1]:
from __future__ import print_function


import matplotlib
matplotlib.use('pdf')




import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp

print(os.getcwd())

plt.rc('lines',linewidth=0.5)

/Users/BenKaiser/Desktop/radial_velocity_calculations


### Ok I need to rewrite this copy of the original notebook to do the same-ish thing with the spectrum of SDSS J1636+1619 (and omitting the model spectrum, presumably).

Now I'm going to try to import plot_spec.py, which is more dangerous because it has executable things in it normally. I tried to comment out the file_setting so I think that means it won't actually "do" anything...

In [2]:
import plot_spec as ps

all_avg


Oh, sweet, I just had to define it to be a string I hadn't defined. We're looking good now I think. The next major hurdle will be that the default plotting method is to put them into plt.whatever instead of fig.whatever, so it's not building a figure object as far as I know, which is what one would want.

In [3]:
plt.plot([1,1])
plt.show()

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:2: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  


Ok, that also answers my question as to whether I have to mess with some setting to have plots display inline.

In [4]:
target_dir= '/Users/BenKaiser/Desktop/Gemini_observing/GN_2020A_DD_114/'
target_spec_file='ravg_fwctb.SDSSJ1636p1619_gemini_600B.fits'
#target_spec_file2='ravg_fwctb.GaiaJ1644m0449_20190825_tellcorr_400m2.fits'
model_spec_file='J1636_CaLi1.50.txt'
#target_spec_file= glob(target_dir+target_spec_file)
#model_spec_file= glob(target_dir+model_spec_file)
print(target_spec_file)
#print(model_spec_file)

ravg_fwctb.SDSSJ1636p1619_gemini_600B.fits


I guess I have to just change the working directory instead of using a long file path for whatever reason...

In [5]:
figure_output_dir='/Users/BenKaiser/Desktop/'

In [6]:
os.chdir(target_dir)

In [7]:
target_spec, header, target_noise= spt.retrieve_spec(target_spec_file)
#target_spec, header, target_noise= spt.retrieve_sdss_spec(target_spec_file)
#target_spec2,header2, target_noise2=spt.retrieve_spec(target_spec_file2)

In [8]:
model_spec= spt.retrieve_model_spec(model_spec_file)

In [9]:
plt.plot(model_spec[0],model_spec[1])
plt.xlim([3700,9000])
plt.show()

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:3: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  This is separate from the ipykernel package so we can avoid doing imports until


And unfortunately, Simon sent me the spectra in units of f_nu instead of f_lambda, which means... I need to convert it most likely... And apparently, I never actually made a function to do the conversion from f_nu to f_lambda. I only have the one to go the other way. Fuck me.

Well, actually let's try plotting the flambda_to_fnu() version of the SDSS spectrum with the one model and see what happens.

In [10]:
#dlambda=target_spec[0][-3]-target_spec[0][-4]
#plt.plot(target_spec[0],spt.flambda_to_fnu(target_spec,dlambda=dlambda)[1])
plt.plot(target_spec[0],target_spec[1])


plt.plot(model_spec[0],model_spec[1])

plt.xlim([4800,8000])
plt.ylim([0,3])
plt.show()

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:10: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  # Remove the CWD from sys.path while we load stuff.


Something looks weird there, but the hell of it is that I found the following STSCI link https://www.stsci.edu/ftp/documents/p2pi/P2PI/ch03_targ_fixed10.html and it matches the inversion of my own formula in the limit in which $\Delta\lambda \rightarrow 0$ or equivalently $\lambda \gg \Delta \lambda$

So screw it. Presumably this is just the culmination of the crappy BOSS relative flux calibration that Hollands et al. 2017 specifically called out for needing to correct. So on that point, time to convert the fluxes from f_nu to f_lambda.

I want to smooth the target spectrum slightly to make it easier to see since it's so noisy

In [11]:
first_mask=[5940,5991]
second_mask=[7025,7074]

clean_target_spec=spt.clean_spectrum(target_spec,np.nanmin(target_spec[0]),np.nanmax(target_spec[0]),[first_mask,second_mask])

In [12]:
#sm_target_spec= ps.convolve_spectrum(target_spec, header, kernel_type='box', pix_width=3)
#sm_target_spec2=ps.convolve_spectrum(target_spec2, header2, kernel_type='box',pix_width=3)
#sm_target_spec= ps.convolve_spectrum(target_spec, header, kernel_type='box', pix_width=5)
sm_target_spec= ps.convolve_spectrum(clean_target_spec, header, kernel_type='box', pix_width=5)
#sm_target_spec=np.copy(target_spec)
#sm_target_spec2=ps.convolve_spectrum(target_spec2, header2, kernel_type='gaussian',pix_width=header2['SEE_SIG'])

In [13]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,5))
label_pos=0.25
label_off=10
#plt.plot(model_spec[0],model_spec[1], label='Best Fit Model')
plt.plot(sm_target_spec[0],sm_target_spec[1], label="SDSS J1636+1619", color='k')
plt.xlim(np.nanmin(target_spec[0]), np.nanmax(target_spec[0]))
#plt.ylim(-0.05, 0.3)
plt.axvline(x=5889.950866765008, linestyle='--', color='r')
plt.axvline(x=5895.9241497669427, linestyle='--', color='r')
plt.text(5895.9241497669427+label_off, label_pos,'Na I D', color='r')

plt.axvline(x=6707.7580436285689, linestyle='--', color='g')
plt.axvline(x=6707.9080032878719, linestyle='--', color='g')
plt.text(6707.9080032878719+label_off, label_pos,'Li I', color='g')

plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

/Users/BenKaiser/Desktop/radial_velocity_calculations/spec_plot_tools.py:501: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Ok, figuring out where the cuts should be on the spectrum to produce the 3 discrete locations of the flux values instead of the current continuous lines over the segment gaps we currently have. I think if I can identify the biggest wavelength jumps in the spectrum, that will tell me where I need to place limits.

In [14]:
trim_one=[4914.3,5942.851]
trim_two=[5985.98,7026.975]
trim_three=[7073.145,7950.]

sm_target_spec1=spt.clean_spectrum(sm_target_spec,trim_one[0],trim_one[1],[])
sm_target_spec2=spt.clean_spectrum(sm_target_spec,trim_two[0],trim_two[1],[])
sm_target_spec3=spt.clean_spectrum(sm_target_spec,trim_three[0],trim_three[1],[])

In [15]:

spt.initiate_science_plot()
#fig=plt.figure(figsize=(7.25,7.25*2./3),constrained_layout=True)
spt.start_ApJ_fig(width_cols=2,constrained_layout=True, width_height=[1.,0.4])
#spt.start_ApJ_fig(width_cols=2,constrained_layout=True, width_height=[1.,0.2])

spt.show_plot(show_legend=False,actually_show=False,show_label=False,line_id='cool_wd',convert_to_air=True)
label_pos=3
label_pos2= 4
label_off=80
#plt.plot(model_spec[0],model_spec[1], label='Best Fit Model')
#plt.plot(sm_target_spec[0],sm_target_spec[1], label="SDSS J1636+1619", color='k')

#plt.plot(sm_target_spec[0],sm_target_spec[1], label="", color='k')
plt.plot(sm_target_spec1[0],sm_target_spec1[1], label="", color='k')
plt.plot(sm_target_spec2[0],sm_target_spec2[1], label="", color='k')
plt.plot(sm_target_spec3[0],sm_target_spec3[1], label="", color='k')



legend_lines=['Li','Na','MgH','K','Ca']
for element in legend_lines:
    plt.axvline(x=-1000.,linestyle='--',color=cp.line_color_dict[element],label=element)


plt.xlim(np.nanmin(target_spec[0])+5, 7800)
#plt.ylim(-0.4,3.5)
#plt.ylim(-0.05, 0.25)
plt.ylim(1,3.3)

k_spot=np.mean([7664.899016,7698.96445153])




k_spot=np.mean([7664.899016,7698.96445153])


#plt.annotate('Na I D',xy=(5895.9241497669427, 2.5),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Li I',xy=(6707.9080032878719, 2.3),xytext=(6707.9080032878719-label_off+40, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca II\nH & K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-120, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.16),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))
#plt.annotate('MgH band',xy=(5190, 0.16),xytext=(5190-210, label_pos-0.05), arrowprops=dict(arrowstyle='-['))
#plt.annotate('K I',xy=(k_spot, 2.1),xytext=(k_spot-40, label_pos), arrowprops=dict(arrowstyle='-['))



plt.ylabel(r'Flux (10$^{-16}$ erg cm$^{-2}$ s$^{-1}$ $\mathrm{\AA}^{-1}$)') 
plt.xlabel(r'Wavelength $(\mathrm{\AA})$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
#spt.show_plot(show_legend=False,actually_show=False,show_label=False,line_id='cool_wd',convert_to_air=True)
plt.legend(loc='upper right',framealpha=1)



print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]

#plt.legend(loc='lower right')
plt.savefig('J1636_spec_'+time_string+'.pdf')#plt.grid(True)

plt.show()
#spt.show_plot(line_id='')

/Users/BenKaiser/Desktop/Goodman_ref_files/line_lists/all_lines_cool_WD.csv
wavelengths.min:  5891.583264
new_wavelengths.min(): 5889.950866765008
wavelengths.min:  5897.558147
new_wavelengths.min(): 5895.924149766943
wavelengths.min:  6709.61
new_wavelengths.min(): 6707.758043628569
wavelengths.min:  6709.76
new_wavelengths.min(): 6707.908003287872
wavelengths.min:  7667.008906
new_wavelengths.min(): 7664.899016001414
wavelengths.min:  7701.083536
new_wavelengths.min(): 7698.9644515250075
wavelengths.min:  8185.5054
new_wavelengths.min(): 8183.255515614788
wavelengths.min:  8197.0434
new_wavelengths.min(): 8194.790398371782
wavelengths.min:  8197.0766
new_wavelengths.min(): 8194.82358940196
wavelengths.min:  3969.59
new_wavelengths.min(): 3968.4672118153667
wavelengths.min:  3934.77
new_wavelengths.min(): 3933.6562946887625
wavelengths.min:  4227.92
new_wavelengths.min(): 4226.729580953195
wavelengths.min:  8500.35
new_wavelengths.min(): 8498.015025790284
wavelengths.min:  8544.44
new

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:66: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [16]:
spt.show_plot(line_id='alkali', convert_to_air=True)

/Users/BenKaiser/Desktop/Goodman_ref_files/line_lists/alkali_lines_vac.csv
wavelengths.min:  5891.583264
new_wavelengths.min(): 5889.950866765008
wavelengths.min:  5897.558147
new_wavelengths.min(): 5895.924149766943
wavelengths.min:  6709.61
new_wavelengths.min(): 6707.758043628569
wavelengths.min:  6709.76
new_wavelengths.min(): 6707.908003287872
wavelengths.min:  7667.008906
new_wavelengths.min(): 7664.899016001414
wavelengths.min:  7701.083536
new_wavelengths.min(): 7698.9644515250075
wavelengths.min:  8185.5054
new_wavelengths.min(): 8183.255515614788
wavelengths.min:  8197.0434
new_wavelengths.min(): 8194.790398371782
wavelengths.min:  8197.0766
new_wavelengths.min(): 8194.82358940196


/Users/BenKaiser/Desktop/radial_velocity_calculations/spec_plot_tools.py:501: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [17]:
plt.rc('font',size=10)
fig= plt.figure(figsize=(4.72,2.44)) #from v1 description image size
label_pos=0.27
label_pos2= 0.15
label_off=110
plt.plot(model_spec[0],model_spec[1], label='Best Fit Model')
plt.plot(sm_target_spec[0],sm_target_spec[1], label="GaiaJ1644-0449 400M1", color='k')
plt.xlim(np.nanmin(target_spec[0]), np.nanmax(target_spec[0]))
plt.ylim(-0.05, 0.3)


plt.annotate('Na I D',xy=(5895.9241497669427, 0.1),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Li I',xy=(6707.9080032878719, 0.2),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca II\nH&K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-80, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.15),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))





plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

/Users/BenKaiser/Desktop/radial_velocity_calculations/spec_plot_tools.py:501: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Well that didn't work... and I don't think that it actually will....

However, in the word doc it says the image original height was 4.61" and width was 8.92", so...

In [18]:
print(4.61/5.)
print(8.92/10.)

0.922
0.892


I guess there's some sort of margin that gets trimmed...

In [19]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,10))
label_pos=0.25
label_off=10
plt.plot(model_spec[0],model_spec[1], label='Best Fit Model')
plt.plot(target_spec[0],target_spec[1], label="GaiaJ1644-0449 400M1", color='k')
plt.xlim(6500, 6800)
plt.ylim(0.1, 0.25)
#plt.axvline(x=5889.950866765008, linestyle='--', color='r')
#plt.axvline(x=5895.9241497669427, linestyle='--', color='r')
#plt.text(5895.9241497669427+label_off, label_pos,'Na I D', color='r')

plt.axvline(x=6707.7580436285689, linestyle='--', color='g')
plt.axvline(x=6707.9080032878719, linestyle='--', color='g')
plt.text(6707.9080032878719+label_off, label_pos,'Li I', color='g')

plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

/Users/BenKaiser/Desktop/radial_velocity_calculations/spec_plot_tools.py:501: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [20]:
plt.rc('font',size=10)
fig= plt.figure(figsize=(15,5)) #from v1 description image size
label_pos=0.27
label_pos2= 0.15
label_off=110
plt.plot(model_spec[0],model_spec[1], label='Best Fit Model')
plt.plot(sm_target_spec[0],sm_target_spec[1], label="GaiaJ1644-0449 400M1", color='k')
plt.plot(sm_target_spec2[0],sm_target_spec2[1],label="GaiaJ1644-0449 400M2", color='grey')
plt.xlim(np.nanmin(target_spec[0]), np.nanmax(target_spec2[0]))
plt.ylim(-0.05, 0.3)


plt.annotate('Na I D',xy=(5895.9241497669427, 0.1),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Li I',xy=(6707.9080032878719, 0.2),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca II\nH&K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-80, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.15),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))





plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

NameError: name 'target_spec2' is not defined

In [ ]:
plt.rc('font',size=10)
fig= plt.figure(figsize=(10,5)) #from v1 description image size
label_pos=0.27
label_pos2= 0.15
label_off=110
plt.plot(model_spec[0],model_spec[1], label='Best Fit Model')
plt.plot(sm_target_spec[0],sm_target_spec[1], label="GaiaJ1644-0449 400M1", color='k')
plt.plot(sm_target_spec2[0],sm_target_spec2[1],label="GaiaJ1644-0449 400M2", color='grey')
plt.xlim(5500,6250)
plt.ylim(0.0, 0.2)


plt.annotate('Na I D',xy=(5895.9241497669427, 0.1),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Li I',xy=(6707.9080032878719, 0.2),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca II\nH&K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-80, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.15),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))





plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

In [ ]:
trim_spot=6800
tsm_target_spec=spt.clean_spectrum(sm_target_spec,np.min(sm_target_spec[0]), trim_spot,[])
tsm_target_spec2=spt.clean_spectrum(sm_target_spec2,trim_spot, np.max(sm_target_spec2[0]),[])
tmodel_spec=spt.clean_spectrum(og_model, np.nanmin(target_spec[0]), np.nanmax(target_spec2[0]),[])

In [ ]:
plt.rc('font',size=10)
fig= plt.figure(figsize=(15,10)) #from v1 description image size
label_pos=0.27
label_pos2= 0.15
label_off=110
plt.plot(tsm_target_spec[0],tsm_target_spec[1], label="GaiaJ1644-0449 400M1 & 400M2", color='k')
plt.plot(tsm_target_spec2[0],tsm_target_spec2[1], color='k')
plt.plot(tmodel_spec[0],tmodel_spec[1], label='Best Fit Model')
plt.xlim(np.nanmin(target_spec[0]), 8500)
plt.ylim(-0.05, 0.3)

k_spot=np.mean([7664.899016,7698.96445153])


plt.annotate('Na I D',xy=(5895.9241497669427, 0.1),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Li I',xy=(6707.9080032878719, 0.2),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca II\nH&K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-80, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.15),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))
plt.annotate('K I',xy=(k_spot, 0.25),xytext=(k_spot-70, label_pos), arrowprops=dict(arrowstyle='-['))






plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

In [ ]:
plt.rc('font',size=10)
fig= plt.figure(figsize=(5,10)) #from v1 description image size
label_pos=0.27
label_pos2= 0.15
label_off=110
plt.plot(tsm_target_spec[0],tsm_target_spec[1], label="GaiaJ1644-0449 400M1 & 400M2", color='k')
plt.plot(tsm_target_spec2[0],tsm_target_spec2[1], color='k')
plt.plot(tmodel_spec[0],tmodel_spec[1], label='Best Fit Model')
plt.xlim(7500,7800)
plt.ylim(0.1, 0.3)


#plt.annotate('Na I D',xy=(5895.9241497669427, 0.1),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Li I',xy=(6707.9080032878719, 0.2),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca II\nH&K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-80, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.15),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))





plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

## 2020-05-29 Because I'm an idiot I want to mess with plotting the unsmoothed spectrum versus the model. fucking moron.

In [ ]:
trim_spot=6800
tsm_target_spec=spt.clean_spectrum(target_spec,np.min(target_spec[0]), trim_spot,[])
tsm_target_spec2=spt.clean_spectrum(target_spec2,trim_spot, np.max(target_spec2[0]),[])
tmodel_spec=spt.clean_spectrum(og_model, np.nanmin(target_spec[0]), np.nanmax(target_spec2[0]),[])

In [ ]:
plt.rc('font',size=10)
fig= plt.figure(figsize=(15,10)) #from v1 description image size
label_pos=0.27
label_pos2= 0.15
label_off=110
plt.plot(tsm_target_spec[0],tsm_target_spec[1], label="GaiaJ1644-0449 400M1 & 400M2", color='k')
plt.plot(tsm_target_spec2[0],tsm_target_spec2[1], color='k')
plt.plot(tmodel_spec[0],tmodel_spec[1], label='Best Fit Model')
plt.xlim(np.nanmin(target_spec[0]), 8500)
plt.ylim(-0.05, 0.3)

k_spot=np.mean([7664.899016,7698.96445153])


plt.annotate('Na I D',xy=(5895.9241497669427, 0.1),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Li I',xy=(6707.9080032878719, 0.2),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca II\nH&K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-80, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.15),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))
plt.annotate('K I',xy=(k_spot, 0.25),xytext=(k_spot-70, label_pos), arrowprops=dict(arrowstyle='-['))





#plt.xlim(6000,8000)
plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

In [ ]:
plt.rc('font',size=10)
fig= plt.figure(figsize=(15,10)) #from v1 description image size
label_pos=0.27
label_pos2= 0.15
label_off=110
plt.plot(tsm_target_spec[0],tsm_target_spec[1], label="GaiaJ1644-0449 400M1 & 400M2", color='k')
plt.plot(tsm_target_spec2[0],tsm_target_spec2[1], color='k')
plt.plot(tmodel_spec[0],tmodel_spec[1], label='Best Fit Model')
plt.xlim(np.nanmin(target_spec[0]), 8500)
plt.ylim(-0.05, 0.3)

k_spot=np.mean([7664.899016,7698.96445153])


#plt.annotate('Na I D',xy=(5895.9241497669427, 0.1),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Li I',xy=(6707.9080032878719, 0.2),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca II\nH&K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-80, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.15),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))
plt.annotate('K I',xy=(k_spot, 0.25),xytext=(k_spot-70, label_pos), arrowprops=dict(arrowstyle='-['))





plt.xlim(6000,8000)
plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

# 2020-06-15 I want the model to actually match the observation, meaning that since I'm smoothing the observation, I need to smooth the model by the same amount since Simon's stuff previously was fit to the default resolution.

I need to verify that Simon's model spectrum is roughly evenly spaced in wavelength

In [ ]:
plt.plot(og_model[0][1:],(np.roll(og_model[0],1)-og_model[0])[1:])
plt.ylim([-50,10])
plt.xlim([3000,10000])
plt.show()

Ok. So Simon's model points are not evenly spaced...So I guess I have to interpolate them to be evenly spaced and then I can do the convolution

In [ ]:
new_model_res=0.1
new_model_waves=np.arange(3000,10000,new_model_res)
new_model=np.array([new_model_waves,np.interp(new_model_waves,og_model[0],og_model[1])])

In [ ]:
plt.plot(og_model[0],og_model[1])
plt.plot(new_model[0],new_model[1], linestyle=':')
plt.xlim([6600,6800])
plt.show()

Ok. So the interpolated version appears to overlay the original model spectrum well enough.

In [ ]:
obs_dlambda=target_spec[0][2]-target_spec[0][1]
print(obs_dlambda)

In [ ]:
sm_width=header['SEE_SIG']
model_sm_width=sm_width*obs_dlambda/new_model_res
sm_target_spec= ps.convolve_spectrum(target_spec, header, kernel_type='gaussian', pix_width=sm_width,kernel_width=3.2/0.3)
sm_target_spec2=ps.convolve_spectrum(target_spec2, header2, kernel_type='gaussian',pix_width=sm_width,kernel_width=3.2/0.3)
sm_model=ps.convolve_spectrum(new_model, header, kernel_type='gaussian',pix_width=model_sm_width, kernel_width=3.2/0.3*obs_dlambda/new_model_res)
sm_model=ps.convolve_spectrum(sm_model,header, kernel_type='box',pix_width=obs_dlambda/new_model_res)

In [ ]:
print(sm_width)
print(model_sm_width)
print(3.2/0.3*obs_dlambda/new_model_res)
print(obs_dlambda/new_model_res)

In [ ]:
trim_spot=6800
tsm_target_spec=spt.clean_spectrum(sm_target_spec,np.min(sm_target_spec[0]), trim_spot,[])
tsm_target_spec2=spt.clean_spectrum(sm_target_spec2,trim_spot, np.max(sm_target_spec2[0]),[])
tmodel_spec=spt.clean_spectrum(sm_model, np.nanmin(target_spec[0]), np.nanmax(target_spec2[0]),[])

In [ ]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(15,10)) #from v1 description image size
label_pos=0.27
label_pos2= 0.15
label_off=110
plt.plot(tsm_target_spec[0],tsm_target_spec[1], label="GaiaJ1644-0449 400M1 & 400M2", color='k')
plt.plot(tsm_target_spec2[0],tsm_target_spec2[1], color='k')
plt.plot(og_model[0],og_model[1], label='Original Model')
plt.plot(tmodel_spec[0],tmodel_spec[1], label='Best Fit Model')


plt.xlim(np.nanmin(target_spec[0]), 8500)
plt.ylim(-0.05, 0.3)

k_spot=np.mean([7664.899016,7698.96445153])


#plt.annotate('Na I D',xy=(5895.9241497669427, 0.1),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Li I',xy=(6707.9080032878719, 0.2),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca II\nH&K',xy=(3968.4672118153667, 0.05),xytext=(3968.4672118153667-80, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('Ca I',xy=(4226.7295809531952, 0.05),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.15),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))
#plt.annotate('K I',xy=(k_spot, 0.25),xytext=(k_spot-70, label_pos), arrowprops=dict(arrowstyle='-['))




#plt.xlim([6600,6800])

plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
plt.xlabel(r'$\lambda(\AA)$')
#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='')

Ok. I give up. I do not know what's wrong with my convolutions here, but pretty clearly they are not working the same way that Simon's did before. Therefore, I think it would be prudent to switch the figure to be the unsmoothed version perhaps? No... Chris was content with the smoothed version previously. Why are you messing with this?? Just let it go dumbass.

## Go do the other god damn parts of the paper!